# P140 — MapReduce: procesamiento simplificado de datos en clústeres grandes

## 1. Título y paper

**Paper:** *MapReduce: simplified data processing on large clusters*  
**Autoría:** Jeffrey Dean, Sanjay Ghemawat  
**Año y venue:** 2004 · OSDI 2004 · Communications of the ACM, 51(1), 107–113  
**Nivel:** L1 · **Motor:** `mapreduce`  
**Ficha completa:** [`P140_mapreduce`](../../papers/foundational/P140_mapreduce/README.md)

**Hito:** Reduce el procesamiento distribuido a dos funciones puras y esconde el reparto, la tolerancia a fallos y la recogida de resultados detrás de ellas.

- [doi:10.1145/1327452.1327492](https://doi.org/10.1145/1327452.1327492)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Procesar terabytes en miles de máquinas exigía escribir a mano el particionado, la comunicación, la recuperación de fallos y la agregación. Cada trabajo reimplementaba lo mismo, y la lógica del problema quedaba enterrada bajo la fontanería.
2. Ejecutar una implementación mínima de la propuesta: Dos funciones: **map**, que transforma cada registro en parejas clave-valor, y **reduce**, que agrega todos los valores de una clave. El sistema se encarga del reparto, del movimiento de datos y de reejecutar lo que falle.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P109


## 4. Intuición

Dos funciones puras —map y reduce— y el sistema se encarga del reparto, del movimiento de datos y de reejecutar lo que falle. El modelo es simple; el sesgo de los datos, no.


## 5. Concepto mínimo

```text
map    : registro → (clave, valor)
reduce : (clave, [valores]) → resultado

El trabajo termina cuando termina el ÚLTIMO reductor
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('mapreduce', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué carga recibe el reductor más ocupado con claves uniformes?
2. ¿Y si una clave se lleva el 40 %?
3. ¿Ayuda añadir reductores?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('mapreduce', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('mapreduce', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con claves uniformes, el más cargado recibe **685** frente a un ideal de 625: solo **1,1×**. Con una clave caliente, recibe **2 367** frente a 357 el menos cargado — **3,79×** el ideal. Y como el trabajo termina con el último, ese reductor **es** el tiempo total.


## 10. Comentario pedagógico

Añadir reductores **no arregla nada**: la clave caliente sigue cayendo en uno solo. El sesgo no es un problema de escala sino de partición, y por eso los sistemas reales acaban con particionadores a medida. Lo que sí ayuda, en otra dimensión, es el combinador: agregar en el mapeador antes de mover datos baja el tráfico 2,56×.


## 11. Error o anti-patrón deliberado

Anti-patrón: diagnosticar un trabajo lento añadiendo máquinas.


In [ ]:
print('Si una clave se lleva el 40 % del trafico, duplicar reductores no la parte.')
print('El reductor caliente tarda lo mismo y sigue siendo el tiempo total.')
print('Antes de escalar, mira la distribucion de tus claves.')

## 12. Corrección

Los dos repartos y el efecto del combinador:


In [ ]:
r = run_paper_lab('mapreduce', seed=3)['result']
print('uniforme:', r['caso_uniforme'])
print('sesgado:', r['caso_sesgado'])
print('combinador:', r['efecto_del_combinador'])

## 13. Desafío guiado

Explica por qué el combinador solo aplica si la operación de reducción es asociativa y conmutativa, con un ejemplo de operación que no lo cumple.


In [ ]:
r = run_paper_lab('mapreduce', seed=3)['result']
show(r)

## 14. Desafío autónomo

Mira la distribución de una clave de particionado real de tu sistema. Calcula qué fracción del tráfico se lleva la clave más frecuente.


## 15. Evidencia de aprendizaje

Guarda la fracción y si tu partición actual la tolera.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P140_mapreduce/README.md) · evaluación formal: [`assessments/papers/P140_mapreduce.md`](../../assessments/papers/P140_mapreduce.md)


## 16. Cierre

Cierra la ruta operativa. Lo siguiente es lo que hay que poder responder cuando alguien pregunta por qué el sistema hizo lo que hizo.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
